# Comparação de Modelos — Agente de Monitoramento (ASA)

**Projeto Interdisciplinar · Entrega 1**

*Inteligência Artificial e Aprendizagem de Máquina · FECAP · 5º Semestre 2026*

> Este notebook parte de `base_unificada_asa-final.csv` (já pronta) e treina e compara os **4 algoritmos de classificação** vistos em aula: Regressão Logística, KNN, Árvore de Decisão e Random Forest. Ao final, escolhemos a melhor baseline, explicamos os fatores mais relevantes e geramos o score Verde/Amarelo/Vermelho.

## Bloco 1 — Carregar a base unificada

Restam pouquíssimos nulos residuais (alunos com registro incompleto em `Contato`) — removemos com `dropna()`, já que são poucas linhas (4 de 12.846).

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score, precision_score, recall_score, f1_score, confusion_matrix, classification_report

df = pd.read_csv("base_unificada_asa-final.csv").dropna()

print("Formato:", df.shape)
print("Taxa de evasao:", df["evadiu"].mean().round(3))

## Bloco 2 — Preparar as features

- `ID_ALUNO` sai — é só identificador, não informação.
- `Grau Acadêmico` sai — depois do filtro de ensino superior (feito no guia de unificação), essa coluna quase não varia mais, carrega pouca informação.
- `tem_bolsa` é booleano (`True`/`False`) — convertido pra `0`/`1`.
- As demais categóricas (`Forma de Ingresso`, `Turno`, `Estado Civil`, `Sexo`) viram dummies (Aula 03).

In [ ]:
df_modelo = df.drop(columns=["ID_ALUNO", "Grau Acadêmico"])
df_modelo["tem_bolsa"] = df_modelo["tem_bolsa"].astype(int)

categoricas = ["Forma de Ingresso", "Turno", "Estado Civil", "Sexo"]
df_modelo = pd.get_dummies(df_modelo, columns=categoricas, drop_first=True)

X = df_modelo.drop(columns=["evadiu"])
y = df_modelo["evadiu"]

print(f"Total de features: {X.shape[1]}")
X.columns.tolist()

## Bloco 3 — Treino, teste e normalização

Regressão Logística e KNN usam distância/otimização baseada em gradiente, então normalizamos com `StandardScaler` (evita `ConvergenceWarning` e melhora um pouco a acurácia). Árvore de Decisão e Random Forest não precisam de normalização — dividem por limiar em cada variável, então a escala não afeta o resultado.

In [ ]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42, stratify=y
)

scaler = StandardScaler()
X_train_norm = scaler.fit_transform(X_train)
X_test_norm = scaler.transform(X_test)

print("Treino:", X_train.shape, " Teste:", X_test.shape)

## Bloco 4 — Treinar os 4 algoritmos

In [ ]:
resultados = {}
modelos_treinados = {}

# --- 1. Regressao Logistica (normalizado) ---
lr = LogisticRegression(max_iter=1000, random_state=42)
lr.fit(X_train_norm, y_train)
modelos_treinados["Regressao Logistica"] = lr

# --- 2. KNN (normalizado) ---
knn = KNeighborsClassifier(n_neighbors=5)
knn.fit(X_train_norm, y_train)
modelos_treinados["KNN (k=5)"] = knn

# --- 3. Arvore de Decisao (dados originais) ---
dt = DecisionTreeClassifier(max_depth=6, random_state=42)
dt.fit(X_train, y_train)
modelos_treinados["Arvore de Decisao"] = dt

# --- 4. Random Forest (dados originais) ---
rf = RandomForestClassifier(n_estimators=200, max_depth=10, random_state=42)
rf.fit(X_train, y_train)
modelos_treinados["Random Forest"] = rf

print("4 modelos treinados.")

## Bloco 5 — Avaliar e comparar (acurácia, precisão, recall, F1)

Como a base é desbalanceada (~20% de evasão), olhamos além da acurácia: precisão, recall e F1 mostram melhor o desempenho na classe que importa (evadiu=1).

In [ ]:
def avaliar(nome, modelo, X_te):
    pred = modelo.predict(X_te)
    return {
        "Modelo": nome,
        "Acuracia": accuracy_score(y_test, pred),
        "Precisao": precision_score(y_test, pred),
        "Recall": recall_score(y_test, pred),
        "F1": f1_score(y_test, pred),
    }

linhas = [
    avaliar("Regressao Logistica", lr, X_test_norm),
    avaliar("KNN (k=5)", knn, X_test_norm),
    avaliar("Arvore de Decisao", dt, X_test),
    avaliar("Random Forest", rf, X_test),
]
comparacao = pd.DataFrame(linhas).set_index("Modelo")
comparacao_pct = (comparacao * 100).round(1)
comparacao_pct

In [ ]:
comparacao_pct.plot(kind="bar", figsize=(9,5))
plt.title("Comparacao dos 4 modelos (base de teste)")
plt.ylabel("%")
plt.ylim(0, 100)
plt.xticks(rotation=15)
plt.legend(loc="lower right")
plt.tight_layout()
plt.show()

melhor_modelo_nome = comparacao["F1"].idxmax()
print(f"Melhor modelo por F1: {melhor_modelo_nome}")

**Conclusão da comparação:** o **Random Forest** teve o melhor F1 (70,7%) e a melhor precisão (85,2%) entre os 4, sendo escolhido como baseline. KNN e Regressão Logística ficaram bem atrás em recall/F1 — provavelmente porque a fronteira de decisão real não é bem aproximada por uma reta (Regressão Logística) nem por vizinhos próximos em alta dimensão (KNN), enquanto os modelos baseados em árvore capturam melhor interações não-lineares entre as features.

## Bloco 6 — Matriz de confusão do melhor modelo (Random Forest)

In [ ]:
pred_rf = rf.predict(X_test)
cm = confusion_matrix(y_test, pred_rf)

fig, ax = plt.subplots(figsize=(5,4))
im = ax.imshow(cm, cmap="Greens")
for i in range(2):
    for j in range(2):
        ax.text(j, i, cm[i,j], ha="center", va="center", fontsize=14)
ax.set_xticks([0,1]); ax.set_xticklabels(["Nao evadiu","Evadiu"])
ax.set_yticks([0,1]); ax.set_yticklabels(["Nao evadiu","Evadiu"])
ax.set_xlabel("Predito"); ax.set_ylabel("Real")
ax.set_title("Matriz de confusao -- Random Forest")
plt.tight_layout()
plt.show()

print(classification_report(y_test, pred_rf, target_names=["Nao evadiu","Evadiu"]))

## Bloco 7 — Explicação dos fatores

O documento do projeto pede *"explicação dos fatores"*. Para o Random Forest usamos `feature_importances_`; para a Regressão Logística, os coeficientes (positivo = aumenta a chance de evasão, negativo = diminui).

In [ ]:
importancias = pd.Series(rf.feature_importances_, index=X.columns).sort_values(ascending=False)

plt.figure(figsize=(9,10))
importancias.sort_values().plot(kind="barh")
plt.xlabel("Importancia (Random Forest)")
plt.title("Fatores mais relevantes -- Random Forest")
plt.tight_layout()
plt.show()

print("Top 10 features (Random Forest):")
importancias.head(10)

In [ ]:
coeficientes = pd.Series(lr.coef_[0], index=X.columns).sort_values()

plt.figure(figsize=(9,10))
coeficientes.plot(kind="barh")
plt.xlabel("Coeficiente (impacto na chance de evasao)")
plt.title("Fatores mais relevantes -- Regressao Logistica")
plt.tight_layout()
plt.show()

**Atenção — resultado a levar em consideração:** investigando os coeficientes da Regressão Logística, `pct_parcelas_em_aberto` tem coeficiente **negativo** — o modelo aprendeu que ter parcela "em aberto" está associado a **NÃO** evadir. Isso provavelmente acontece porque parcelas de quem evade tendem a ser **canceladas** no sistema, não ficam "em aberto" — ou seja, a coluna mede em parte "ainda matriculado", não só "dificuldade financeira". Vale documentar isso como limitação no relatório, e considerar refinar a feature (ex.: contar só parcelas vencidas e não pagas, comparando com a data de hoje).

## Bloco 8 — Árvore de Decisão: visualizando as regras (bônus de interpretabilidade)

In [ ]:
plt.figure(figsize=(20,10))
plot_tree(dt, feature_names=X.columns, class_names=["Nao evadiu","Evadiu"],
          filled=True, max_depth=2, fontsize=9)
plt.title("Arvore de Decisao (2 primeiros niveis)")
plt.show()

## Bloco 9 — Score de 0 a 100 e classificação Verde/Amarelo/Vermelho

O documento do projeto pede um **score de 0 a 100** e uma classificação em 3 categorias. Usamos o `predict_proba()` do **Random Forest** (melhor modelo).

⚠️ Os limites `33`/`66` abaixo são ponto de partida — ajustem olhando a distribuição real dos scores de vocês, e validem com o ASA antes de considerar final (é o que o documento do projeto pede: *"limites configuráveis e aprovados pela FECAP/ASA"*).

In [ ]:
probabilidades = rf.predict_proba(X_test)[:, 1]
score = (probabilidades * 100).round(1)

def classificar(s):
    if s < 33:
        return "Verde"
    elif s < 66:
        return "Amarelo"
    else:
        return "Vermelho"

classificacao = [classificar(s) for s in score]

resultado = pd.DataFrame({
    "ID_ALUNO": df.loc[X_test.index, "ID_ALUNO"].values,
    "score": score,
    "classificacao": classificacao,
})
print(resultado.head(10))
print("\n", resultado["classificacao"].value_counts())
print("\n", (resultado["classificacao"].value_counts(normalize=True) * 100).round(1))

In [ ]:
plt.figure(figsize=(8,4))
plt.hist(score, bins=30, color="#0B2E22")
plt.axvline(33, color="orange", linestyle="--", label="limite Verde/Amarelo (33)")
plt.axvline(66, color="red", linestyle="--", label="limite Amarelo/Vermelho (66)")
plt.xlabel("Score de risco de evasao")
plt.ylabel("Numero de alunos")
plt.title("Distribuicao do score -- conjunto de teste")
plt.legend()
plt.tight_layout()
plt.show()

## Bloco 10 — Testar 3 alunos específicos, com perfis diferentes

Pegamos 1 exemplo de cada classificação (Verde, Amarelo, Vermelho) do próprio conjunto de teste, pra ver o modelo aplicado em casos concretos, lado a lado.

In [ ]:
id_verde = resultado[resultado["classificacao"] == "Verde"]["ID_ALUNO"].iloc[0]
id_amarelo = resultado[resultado["classificacao"] == "Amarelo"]["ID_ALUNO"].iloc[0]
id_vermelho = resultado[resultado["classificacao"] == "Vermelho"]["ID_ALUNO"].iloc[0]

ids_exemplo = [id_verde, id_amarelo, id_vermelho]

colunas_interesse = ["ID_ALUNO", "Idade", "tempo_de_curso", "pct_parcelas_em_aberto",
                      "media_notas", "pct_reprovacao", "total_contatos"]

tabela_exemplos = df[df["ID_ALUNO"].isin(ids_exemplo)][colunas_interesse].merge(
    resultado[["ID_ALUNO", "score", "classificacao"]], on="ID_ALUNO"
)
tabela_exemplos = tabela_exemplos.set_index("ID_ALUNO").loc[ids_exemplo].reset_index()

tabela_exemplos

## Bloco 11 — Simular a chegada de 3 alunos novos (não estão na base)

Isso simula o uso real do modelo em produção: um aluno novo chega, vocês preenchem os valores das features na mão, e pedem a previsão.

A função `preparar_aluno_novo` cuida de aplicar os mesmos `get_dummies` e alinhar as colunas com o que o modelo espera.

In [ ]:
def preparar_aluno_novo(dados_dict):
    linha = pd.DataFrame([dados_dict])
    linha["tem_bolsa"] = linha["tem_bolsa"].astype(int)
    linha = pd.get_dummies(linha, columns=categoricas)
    linha = linha.reindex(columns=X.columns, fill_value=0)
    return linha

aluno_A = {
    "tempo_de_curso": 1, "Idade": 19, "mora_sp_capital": 1,
    "total_parcelas": 2, "pct_parcelas_em_aberto": 0.0, "pct_parcelas_acordo": 0.0,
    "atraso_medio_dias": 0.0, "tem_bolsa": False, "valor_total_periodo": 3200.0,
    "media_notas": 85.0, "media_assiduidade": 95.0, "pct_reprovacao": 0.0,
    "tendencia_notas": 0.0, "tendencia_assiduidade": 0.0, "total_semestres": 1,
    "total_contatos": 0, "dias_desde_ultimo_contato": 999,
    "Forma de Ingresso": "Enem", "Turno": "Manhã", "Estado Civil": "Solteiro", "Sexo": "Feminino",
}

aluno_B = {
    "tempo_de_curso": 4, "Idade": 24, "mora_sp_capital": 0,
    "total_parcelas": 20, "pct_parcelas_em_aberto": 0.15, "pct_parcelas_acordo": 0.05,
    "atraso_medio_dias": 8.0, "tem_bolsa": True, "valor_total_periodo": 18000.0,
    "media_notas": 68.0, "media_assiduidade": 78.0, "pct_reprovacao": 0.10,
    "tendencia_notas": -5.0, "tendencia_assiduidade": -3.0, "total_semestres": 4,
    "total_contatos": 2, "dias_desde_ultimo_contato": 180,
    "Forma de Ingresso": "Vestibular", "Turno": "Noite", "Estado Civil": "Solteiro", "Sexo": "Masculino",
}

aluno_C = {
    "tempo_de_curso": 6, "Idade": 31, "mora_sp_capital": 0,
    "total_parcelas": 35, "pct_parcelas_em_aberto": 0.40, "pct_parcelas_acordo": 0.20,
    "atraso_medio_dias": 25.0, "tem_bolsa": False, "valor_total_periodo": 28000.0,
    "media_notas": 45.0, "media_assiduidade": 60.0, "pct_reprovacao": 0.35,
    "tendencia_notas": -18.0, "tendencia_assiduidade": -15.0, "total_semestres": 6,
    "total_contatos": 6, "dias_desde_ultimo_contato": 20,
    "Forma de Ingresso": "Transferência", "Turno": "Noite", "Estado Civil": "Casado(a)", "Sexo": "Masculino",
}

alunos_novos = {"Aluno A (calouro, tudo em dia)": aluno_A,
                "Aluno B (alguns atrasos, notas caindo)": aluno_B,
                "Aluno C (varios atrasos, reprovacoes, contatos frequentes)": aluno_C}

for nome, dados in alunos_novos.items():
    linha_preparada = preparar_aluno_novo(dados)
    prob = rf.predict_proba(linha_preparada)[0][1]
    score_novo = round(prob * 100, 1)
    print(nome)
    print("  Score (Random Forest):", score_novo, "->", classificar(score_novo))
    print()

## Bloco 12 — Salvar os resultados (opcional, para anexar ao relatório)

In [ ]:
comparacao_pct.to_csv("comparacao_modelos.csv")
resultado.to_csv("scores_conjunto_teste.csv", index=False)
importancias.to_csv("importancia_features_rf.csv", header=["importancia"])
print("Arquivos salvos: comparacao_modelos.csv, scores_conjunto_teste.csv, importancia_features_rf.csv")

---
## Resumo para o Relatório Técnico

| Modelo | Acurácia | Precisão | Recall | F1 |
|---|---|---|---|---|
| Regressão Logística | 84,2% | 67,5% | 37,2% | 48,0% |
| KNN (k=5) | 84,8% | 68,9% | 40,6% | 51,1% |
| Árvore de Decisão | 88,6% | 73,8% | 64,8% | 69,0% |
| **Random Forest (escolhido)** | **90,2%** | **85,2%** | **60,4%** | **70,7%** |

- **Base:** 12.842 alunos, 30 features (após dummies), split 80/20 estratificado.
- **Melhor modelo:** Random Forest (200 árvores, profundidade máxima 10) — melhor F1 e melhor precisão.
- **Fatores mais relevantes (RF):** `valor_total_periodo`, `media_notas`, `dias_desde_ultimo_contato`, `pct_parcelas_em_aberto`, `tempo_de_curso`, `total_parcelas`.
- **Limitação identificada:** `pct_parcelas_em_aberto` tem coeficiente negativo na Regressão Logística — provável efeito de parcelas de quem evade serem canceladas (não ficarem "em aberto"). Documentar e refinar na Entrega 2.
- **Score:** `predict_proba()` × 100, limites 33/66 (Verde/Amarelo/Vermelho) — ainda a validar com o ASA.

## Próximos passos

1. Ajustar os limites Verde/Amarelo/Vermelho com base na distribuição real dos scores (quartis).
2. Validação cruzada e tuning de hiperparâmetros dos 4 modelos.
3. Testar balanceamento de classes (class_weight="balanced" / oversampling), dado o desbalanceamento (19,6% de evasão).
4. Refinar `pct_parcelas_em_aberto` (separar parcelas vencidas de parcelas dentro do prazo).
5. Documentar tudo no Relatório Técnico e no Model Card (Entrega 1) e evoluir para explicabilidade (SHAP) na Entrega 2.